# Jina Reranker Structure tr?n Colab

Notebook n?y ch?y `src/re-ranker/jina_rerank_structure.py` v?i model `jinaai/jina-reranker-v2-base-multilingual` tr?n input structure.


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa bật GPU")

CUDA available: True
GPU: Tesla T4


In [2]:
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" accelerate sentence-transformers einops

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content

!rm -rf Text-Mining---RAG-on-News
!git clone -b Alibaba https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git

%cd /content/Text-Mining---RAG-on-News

/content
Cloning into 'Text-Mining---RAG-on-News'...
remote: Enumerating objects: 529, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 529 (delta 110), reused 112 (delta 45), pack-reused 344 (from 1)
Receiving objects: 100% (529/529), 21.41 MiB | 20.76 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/Text-Mining---RAG-on-News


In [4]:
import torch
import transformers
import importlib.util

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("Jina reranker dependencies ready")

CUDA available: True
GPU: Tesla T4
transformers: 4.44.2
Jina reranker dependencies ready


In [5]:
from pathlib import Path

input_path = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")

print("Input exists:", input_path.exists())
print("Input path:", input_path)

if input_path.exists():
    with input_path.open("r", encoding="utf-8") as f:
        first_line = f.readline()
    print(first_line[:500])

Input exists: True
Input path: /content/drive/MyDrive/out_embedding/per_query_structured.jsonl
{"qa_id": "211640_1", "qa_type": "factoid", "question": "Những loại nội tạng động vật nào được khuyến cáo nên hạn chế để tránh tăng axit uric và hại thận?", "gold_articles": ["211640"], "top_articles": ["211640", "211640", "28856", "28856", "207094", "205401", "28348", "31222", "27887", "27887"], "candidates": [{"rank": 1, "chunk_index": 1, "chunk_id": "211640_structured_0001", "article_id": "211640", "score": 0.884665, "text": "Tiêu đề: Không muốn hại thận, cần hạn chế 4 loại thịt\nMô tả: Purin


In [6]:
from pathlib import Path

script_path = Path("src/re-ranker/jina_rerank_structure.py")

print("Script exists:", script_path.exists())
print("Script path:", script_path)


Script exists: True
Script path: src/re-ranker/jina_rerank_structure.py


In [7]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/out_reranker")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output dir exists:", output_dir.exists())

Output dir exists: True


## Hugging Face

Model Jina public th??ng kh?ng c?n login. N?u g?p l?i gated/private, ch?y `from huggingface_hub import login; login()`.


In [ ]:
from huggingface_hub import login

login("hf_................")

In [ ]:
!python src/re-ranker/jina_rerank_structure.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_test.jsonl" \
    --limit 5 \
    --batch-size 8 \
    --max-length 1024

In [ ]:
import json
from pathlib import Path

test_output = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_test.jsonl")

print("Test output exists:", test_output.exists())

with test_output.open("r", encoding="utf-8") as f:
    row = json.loads(next(f))

print("QA ID:", row["qa_id"])
print("Question:", row["question"])
print("Embedding type:", row["embedding_type"])
print("Reranker:", row["reranker"])
print("Metrics:", row["rerank_metrics"])
print("Top 1 score:", row["reranked_candidates"][0]["rerank_score"])
print("Top 1 text:", row["reranked_candidates"][0]["text"][:300])

In [9]:
!python src/re-ranker/jina_rerank_structure.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl" \
    --batch-size 8 \
    --max-length 1024

config.json: 1.10kB [00:00, 2.07MB/s]
configuration_xlm_roberta.py: 2.73kB [00:00, 13.0MB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_xlm_roberta.py: 43.8kB [00:00, 67.9MB/s]
xlm_padding.py: 9.82kB [00:00, 35.4MB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual:
- xlm_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
embedding.py: 2.56kB [00:00, 11.7MB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual:
- embedding.py
. Make sure to double-check they do not conta

In [10]:
from pathlib import Path

input_file = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")
output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl")

def count_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print("Input lines:", count_jsonl(input_file))
print("Output lines:", count_jsonl(output_file))

Input lines: 114
Output lines: 114


In [11]:
import json
import pandas as pd
from pathlib import Path

output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl")
summary_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_summary.csv")

rows = []

with output_file.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        rows.append(row["rerank_metrics"])

df = pd.DataFrame(rows)

summary = {
    "config": "structure_jina_reranker",
    "num_queries": len(df),
    "hit@1": df["hit@1"].mean(),
    "hit@5": df["hit@5"].mean(),
    "recall@5": df["recall@5"].mean(),
    "mrr@5": df["mrr@5"].mean(),
    "ndcg@5": df["ndcg@5"].mean(),
}

summary_df = pd.DataFrame([summary])


summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
print(f"Saved summary to {summary_file}")
summary_df

Saved summary to /content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_summary.csv


,config,num_queries,hit@1,hit@5,recall@5,mrr@5,ndcg@5
0,structure_jina_reranker,114,0.649123,0.684211,0.684211,0.666667,0.671261
